### **Assignment 1: Finding the Best Delivery Promise for Rosa's Pizza**


**This notebook analyzes Rosa's delivery data to measure how often orders are late (Part I), find the most profitable delivery promise for each zone and time block (Part II), and support a web app for Rosa (Part III).**

In [1]:
!pip install -q git+https://github.com/zhouy185/rosa-starter.git

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


**Setup:** Install the starter package, import the data and functions, and check that the simulator works. All calls use `seed=1` so results are the same every run.


In [2]:
import numpy as np
from starter import ZONES, TIME_BLOCKS, COSTS, PROMISE, delivery_times

In [3]:
times = delivery_times("Far West", "Fri/Sat eve", 45, seed=1)
print(len(times), "orders; average delivery time", round(times.mean(), 1),
"minutes")

210 orders; average delivery time 42.9 minutes


### **How Bad Is It?**

### **Part I: (a)**

In [10]:

import numpy as np
from starter import ZONES, TIME_BLOCKS, delivery_times

def get_late_percentage(zone, time_block, promise, seed=1):

    # CASE 1: Both zone and time block are specific names
    if zone != 'all' and time_block != 'all':
        times = delivery_times(zone, time_block, promise, seed=1)
        return (times > promise).mean() * 100

    # CASE 2: Zone is 'all', but time block is specific
    if zone == 'all' and time_block != 'all':
        basket = []
        for z in ZONES:
            times = delivery_times(z, time_block, promise, seed=1)
            basket.append(times)

        data = np.concatenate(basket)
        return (data > promise).mean() * 100

    # CASE 3: Specific zone, but time block is 'all'
    if zone != 'all' and time_block == 'all':
        basket = []
        for tb in TIME_BLOCKS:
            times = delivery_times(zone, tb, promise, seed=1)
            basket.append(times)

        data = np.concatenate(basket)
        return (data > promise).mean() * 100

    # CASE 4: Both zone and time block are 'all'
    if zone == 'all' and time_block == 'all':
        basket = []
        for z in ZONES:
            for tb in TIME_BLOCKS:
                times = delivery_times(z, tb, promise, seed=1)
                basket.append(times)

        data = np.concatenate(basket)
        return (data > promise).mean() * 100

### **Testing function**

In [18]:
# Test Case 1: Specific zone and specific time block (e.g., "Far West" and "Fri/Sat eve")
print("Case 1:", get_late_percentage("Far West", "Fri/Sat eve", 45))

# Test Case 2: 'all' zones and a specific time block (e.g., 'all' and "Lunch")
print("Case 2:", get_late_percentage('all', "Lunch", 45))

# Test Case 3: A specific zone and 'all' time blocks (e.g., "North" and 'all')
print("Case 3:", get_late_percentage("North", 'all', 45))

# Test Case 4: Both zone and time block are 'all'
print("Case 4:", get_late_percentage('all', 'all', 45))


Case 1: 36.666666666666664
Case 2: 3.3333333333333335
Case 3: 6.800000000000001
Case 4: 8.291873963515755


### **Part I: (b)**

In [17]:
# Make an empty basket to hold our zone rankings
zone_ranking = []

# Loop through every zone and every time block
for z in ZONES:
    for tb in TIME_BLOCKS:
        # Call the function you wrote with a 45-minute promise
        late_rate = get_late_percentage(z, tb, 45)

        # Save the zone, time block, and its late rate into our list
        zone_ranking.append((z, tb, late_rate))

# Sort the list from highest late rate to lowest (descending order)
zone_ranking_sorted = sorted(zone_ranking, key=lambda x: x[2], reverse=True)

# Print out the results without the pipe character
print("Ranked Late Arrival Rates (descending order):")
for zone, time_block, rate in zone_ranking_sorted:
    print("Zone: " + zone + " - Time Block: " + time_block + " - Late Rate: " + str(rate) + "%")

Ranked Late Arrival Rates (descending order):
Zone: Far West - Time Block: Fri/Sat eve - Late Rate: 36.666666666666664%
Zone: North - Time Block: Fri/Sat eve - Late Rate: 16.666666666666664%
Zone: Far West - Time Block: Weekday eve - Late Rate: 11.176470588235295%
Zone: Central - Time Block: Fri/Sat eve - Late Rate: 6.842105263157896%
Zone: North - Time Block: Weekday eve - Late Rate: 6.0%
Zone: Far West - Time Block: Other - Late Rate: 4.6875%
Zone: Central - Time Block: Weekday eve - Late Rate: 4.242424242424243%
Zone: North - Time Block: Lunch - Late Rate: 3.684210526315789%
Zone: Central - Time Block: Lunch - Late Rate: 3.4615384615384617%
Zone: Central - Time Block: Other - Late Rate: 2.631578947368421%
Zone: North - Time Block: Other - Late Rate: 2.5%
Zone: Far West - Time Block: Lunch - Late Rate: 2.5%


### **Part I: (c)**

In [19]:
def get_avg_delivery_time(zone, time_block, promise, seed=1):

    # CASE 1: Both zone and time block are specific names
    if zone != 'all' and time_block != 'all':
        times = delivery_times(zone, time_block, promise, seed=1)
        return times.mean()

    # CASE 2: Zone is 'all', but time block is specific
    if zone == 'all' and time_block != 'all':
        basket = []
        for z in ZONES:
            times = delivery_times(z, time_block, promise, seed=1)
            basket.append(times)

        data = np.concatenate(basket)
        return data.mean()

    # CASE 3: Specific zone, but time block is 'all'
    if zone != 'all' and time_block == 'all':
        basket = []
        for tb in TIME_BLOCKS:
            times = delivery_times(zone, tb, promise, seed=1)
            basket.append(times)

        data = np.concatenate(basket)
        return data.mean()

    # CASE 4: Both zone and time block are 'all'
    if zone == 'all' and time_block == 'all':
        basket = []
        for z in ZONES:
            for tb in TIME_BLOCKS:
                times = delivery_times(z, tb, promise, seed=1)
                basket.append(times)

        data = np.concatenate(basket)
        return data.mean()

### **Testing function**

In [25]:
# Test Case 1: Specific zone and specific time block
print("Case 1:", get_avg_delivery_time("Far West", "Fri/Sat eve", 45))

# Test Case 2: 'all' zones and a specific time block
print("Case 2:", get_avg_delivery_time('all', "Lunch", 45))

# Test Case 3: A specific zone and 'all' time blocks
print("Case 3:", get_avg_delivery_time("North", 'all', 45))

# Test Case 4: Both zone and time block are 'all'
print("Case 4:", get_avg_delivery_time('all', 'all', 45))

Case 1: 42.9195238095238
Case 2: 28.219298245614034
Case 3: 30.9752
Case 4: 31.193739635157545


### **Part I: (d)**

In [26]:
# Make an empty basket to hold our zone rankings
zone_ranking_avg_delivery = []

# Loop through every zone and every time block
for z in ZONES:
    for tb in TIME_BLOCKS:
        # Call the average delivery time function with a 45-minute promise
        avg_time = get_avg_delivery_time(z, tb, 45)

        # Save the zone, time block, and its average delivery time into our list
        zone_ranking_avg_delivery.append((z, tb, avg_time))

# Sort the list from highest average delivery time to lowest (descending order)
zone_ranking_avg_delivery_sorted = sorted(zone_ranking_avg_delivery, key=lambda x: x[2], reverse=True)

# Print out the results using the pure, explicit long way
print("Ranked Average Delivery Times (descending order):")
for zone, time_block, avg_time in zone_ranking_avg_delivery_sorted:
    print("Zone: " + zone + " - Time Block: " + time_block + " - Avg Time: " + str(avg_time) + " minutes")

Ranked Average Delivery Times (descending order):
Zone: Far West - Time Block: Fri/Sat eve - Avg Time: 42.9195238095238 minutes
Zone: North - Time Block: Fri/Sat eve - Avg Time: 36.76533333333334 minutes
Zone: Far West - Time Block: Weekday eve - Avg Time: 35.044117647058826 minutes
Zone: Central - Time Block: Fri/Sat eve - Avg Time: 32.13263157894737 minutes
Zone: North - Time Block: Weekday eve - Avg Time: 31.3756 minutes
Zone: Far West - Time Block: Lunch - Avg Time: 30.58083333333333 minutes
Zone: Far West - Time Block: Other - Avg Time: 30.04010416666667 minutes
Zone: Central - Time Block: Weekday eve - Avg Time: 29.030909090909088 minutes
Zone: North - Time Block: Lunch - Avg Time: 28.672631578947364 minutes
Zone: North - Time Block: Other - Avg Time: 27.655624999999997 minutes
Zone: Central - Time Block: Lunch - Avg Time: 26.798076923076923 minutes
Zone: Central - Time Block: Other - Avg Time: 26.03684210526316 minutes


### **Part I: (e)**:

### **Is late rate or Avg Delivery Time measure better for Rosa's business decison making?**

The late rate (b) is more relevant to Rosa's decisions. Based on our analysis, the longest average delivery time is 42.9 minutes, which is technically still under the 45-minute promise. However, that 42.9-minute average also means that 36.7% of her customers are getting their orders late, which is quite a lot. From a customer's perspective, the late rate measures how often Rosa breaks her brand promise, making it a core metric for business decisions.


### **PART II: Exploring The Best Promise**

### **Part II: (a)**

In [43]:
def calculate_late_cost(costs):
    # Direct refund cost for a late order
    refund_cost = costs["refund"]

    # Churn cost: future lost orders multiplied by profit margin per order
    churn_cost = costs["churn_orders"] * costs["margin"]

    # Total cost per late order
    total_cost = refund_cost + churn_cost
    return total_cost

# Test the function using the COSTS dictionary and print the output
cost_per_late = calculate_late_cost(COSTS)
print("Total cost per late order: $" + str(cost_per_late))

Total cost per late order: $26.2


### **Part II: (b)**: **How to Properly choose the Range?**

If we were to consider a range from 5 to 60 minutes in increments of 5 minutes, 10 to 25 minutes would be too little time for delivery. Even though a shorter delivery time seems very appealing to customers, Rosa's business would suffer deeply: order volumes exceeding the pizzeria's capacity would result in massive complaints, refund costs, and customer churn.

Choosing a range from 30 to 90 minutes in 5-minute increments is the best option because it allows Rosa to find the exact balance, maintaining a high volume of customer orders to generate revenue while keeping promises tight enough to control late-arrival refund requests. For a range to be good, the best promise must fall inside it and not at the extremes of it.

Conversely, anything over 60 minutes would ensure on-time delivery and little-to-no refund or churn costs, but long delivery times would make the pizzeria unappealing to customers, dropping order volume. We still include 65 to 90 minutes to confirm that profit keeps falling past the peak.

The results confirm the range is right: net profit rises, peaks at 55 minutes ($1,212.40), and then falls.

### **Function to Choose the Best Promise**

In [37]:
def find_best_promise(zone, time_block, promise_list, costs, seed=1):
    # Calculate total cost per late order using our previous logic
    cost_per_late = calculate_late_cost(costs)

    best_promise = None
    max_net_profit = -999999.0

    for p in promise_list:
        # Simulate delivery times for this specific promise
        times = delivery_times(zone, time_block, promise=p, seed=1)
        n_orders = len(times)

        if n_orders == 0:
            net_profit = 0.0
        else:
            # Count how many orders arrived after the promised time (late)
            n_late = sum(1 for t in times if t > p)

            # Gross profit collected from all orders
            gross_profit = n_orders * costs["margin"]

            # Total late costs incurred
            total_late_cost = n_late * cost_per_late

            # Net profit = Gross profit minus late costs
            net_profit = gross_profit - total_late_cost

        # Track the promise that yields the highest net profit
        if net_profit > max_net_profit:
            max_net_profit = net_profit
            best_promise = p

    return best_promise, max_net_profit

### **Testing Function**

In [45]:

# Define the range of promises to test in 5-minute steps
promises_to_try = [30, 35, 40, 45, 50, 55, 60, 65, 70, 75, 80, 85, 90]

# Test the function on "Far West" and "Fri/Sat eve"
zone_test = "Far West"
time_block_test = "Fri/Sat eve"

best_p, profit = find_best_promise(zone_test, time_block_test, promises_to_try, COSTS)

# Print out the results using explicit string concatenation
print("Recommended Promise for " + zone_test + " (" + time_block_test + "): " + str(best_p) + " minutes")
print("Expected Net Profit: $" + str(round(profit, 2)))

Recommended Promise for Far West (Fri/Sat eve): 55 minutes
Expected Net Profit: $1212.4


### **PART III: Creating a Web App**